# 03 - Postulate 2: Evolution (Unitary Dynamics)

**The evolution of a closed quantum system is described by a unitary transformation.**

In plain English: quantum states change over time by being multiplied by unitary matrices. That is what quantum gates are. Every gate is a unitary matrix. Every unitary matrix is a valid gate.

This postulate also says: quantum evolution is **reversible**. For every gate U, there exists U^dagger that undoes it. You can always go back. This is fundamentally different from classical computing where some operations (like AND) lose information.

In [ ]:
import numpy as np

## Unitary = Reversible = Probability Preserving

A unitary matrix U has three equivalent properties:
1. U @ U^dagger = I (reversible -- U^dagger undoes U)
2. It preserves the norm of vectors (probabilities still add to 1 after the gate)
3. It preserves inner products (distinguishable states stay distinguishable)

These are all the same thing said differently.

In [ ]:
# Hadamard gate
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

# Property 1: Reversible
print("H @ H^dagger =")
print(np.round(H @ H.conj().T, 10))

# Property 2: Preserves norm
psi = np.array([np.sqrt(0.7), np.sqrt(0.3)], dtype=complex)
psi_after = H @ psi
print(f"\nNorm before H: {np.linalg.norm(psi):.4f}")
print(f"Norm after H:  {np.linalg.norm(psi_after):.4f}")

# Property 3: Preserves inner products
phi = np.array([1, 0], dtype=complex)
ip_before = np.dot(psi.conj(), phi)
ip_after = np.dot((H @ psi).conj(), H @ phi)
print(f"\nInner product before H: {ip_before:.4f}")
print(f"Inner product after H:  {ip_after:.4f}")

## Time Evolution and the Schrodinger Equation

In physics, quantum states evolve according to the Schrodinger equation:

`i * hbar * d|psi>/dt = H_op |psi>`

Where H_op is the Hamiltonian (the energy operator, not the Hadamard gate).

The solution is: `|psi(t)> = e^(-i * H_op * t / hbar) |psi(0)>`

That exponential of the Hamiltonian is always a unitary matrix. So time evolution IS a unitary gate.

In quantum computing, we do not care about continuous time evolution. We just apply discrete unitary gates. But it is good to know where they come from.

In [ ]:
from scipy.linalg import expm

# Hamiltonian for a qubit in a magnetic field along Z
# H_op = (omega/2) * Z
omega = 1.0  # frequency
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H_op = (omega / 2) * Z

# Time evolution operator: U(t) = e^(-i * H_op * t)
# (setting hbar = 1 for simplicity)
t = np.pi / 2  # evolve for time pi/2
U = expm(-1j * H_op * t)

print(f"Hamiltonian (energy operator):")
print(H_op)
print(f"\nTime evolution U(t={t:.4f}):")
print(U.round(4))
print(f"\nIs U unitary? {np.allclose(U @ U.conj().T, np.eye(2))}")

# Apply to |+> state
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
evolved = U @ ket_plus
print(f"\n|+> evolved by time t={t:.4f}:")
print(f"  {evolved.round(4)}")
print(f"  This is a rotation around the Z axis on the Bloch sphere.")

## Common Quantum Gates as Rotations

Every single qubit gate is a rotation on the Bloch sphere.

- **Rx(theta)**: rotation around X axis by angle theta
- **Ry(theta)**: rotation around Y axis by angle theta
- **Rz(theta)**: rotation around Z axis by angle theta

Any gate can be decomposed into these rotations.

In [ ]:
def Rx(theta):
    return np.array([
        [np.cos(theta/2), -1j*np.sin(theta/2)],
        [-1j*np.sin(theta/2), np.cos(theta/2)]
    ], dtype=complex)

def Ry(theta):
    return np.array([
        [np.cos(theta/2), -np.sin(theta/2)],
        [np.sin(theta/2), np.cos(theta/2)]
    ], dtype=complex)

def Rz(theta):
    return np.array([
        [np.exp(-1j*theta/2), 0],
        [0, np.exp(1j*theta/2)]
    ], dtype=complex)

# Rx(pi) = X gate (up to global phase)
print("Rx(pi):")
print(Rx(np.pi).round(4))

# Ry(pi) = Y gate (up to global phase)
print("\nRy(pi):")
print(Ry(np.pi).round(4))

# Rz(pi) = Z gate (up to global phase)
print("\nRz(pi):")
print(Rz(np.pi).round(4))

# All are unitary
theta = 1.234
for name, gate in [("Rx", Rx(theta)), ("Ry", Ry(theta)), ("Rz", Rz(theta))]:
    print(f"\n{name}({theta:.3f}) unitary? {np.allclose(gate @ gate.conj().T, np.eye(2))}")

## Reversibility: Undoing a Gate

Every unitary gate U can be undone by applying U^dagger (conjugate transpose). This is fundamentally different from classical logic where gates like AND are irreversible.

In [ ]:
# Start with |0>
psi = np.array([1, 0], dtype=complex)
print(f"Start: {psi}")

# Apply some gates
gate1 = Ry(np.pi/3)
gate2 = Rz(np.pi/5)
gate3 = Rx(np.pi/7)

psi = gate1 @ psi
psi = gate2 @ psi
psi = gate3 @ psi
print(f"After 3 gates: {psi.round(4)}")

# Undo them in reverse order
psi = gate3.conj().T @ psi
psi = gate2.conj().T @ psi
psi = gate1.conj().T @ psi
print(f"After undoing: {psi.round(4)}")
print("Back to |0>.")

## No Information is Lost

Because unitary evolution is reversible, quantum computing never loses information (until you measure). This means:
- Every quantum gate has an inverse
- The number of input qubits always equals the number of output qubits
- You cannot create or destroy information, only transform it

Measurement breaks this rule -- which is why it gets its own postulate.

## Exercises

1. Show that Ry(pi/2) applied to |0> gives the state |+> (up to global phase). Verify both give the same probabilities.

2. Create a unitary gate as a product of three rotations: Rz(0.3) @ Ry(0.5) @ Rz(0.7). Verify it is unitary. Apply it to |0> and verify the norm is preserved.

3. The T gate adds a phase of pi/4 to |1>. Write it as a rotation gate. Which axis is it a rotation around?

In [ ]:
# Your code here
